# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/innouguru/flyrank-intenship-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The action queue contains the 50 highest-ranked February 2026 CTR candidates from the validated Random Forest model. The candidates are ranked by `model_score`, which represents the model's predicted probability of future CTR recovery.

The reason code for entering the queue is `CTR_BELOW_POSITION_PEERS`. This identifies pages whose current CTR was below the median CTR of their search-position peer group. The model score is then used to prioritize the candidates within this group.

The model score is used for prioritization, not as an automatic instruction to change a page. The recommended action is designed to support human review.

Pages with **100 or fewer GSC impressions** are assigned `MONITOR`. These pages have limited search exposure, so there may not be enough evidence to justify an immediate intervention. They should accumulate more search evidence before a change is considered.

Pages with **more than 100 GSC impressions** are assigned `SERP_REVIEW`. The first review should focus on search intent and the SERP presentation, particularly the title and snippet. If the SERP review does not explain the low CTR, a human reviewer can consider whether a deeper content refresh is appropriate.

Engagement metrics are provided as supporting context rather than proof of content quality. Content freshness is also a human-review consideration rather than an automated decision rule because content age was not one of the validated model features.

The queue therefore follows this decision path:

**Top 50 model-ranked candidates → check search exposure → MONITOR if impressions ≤100 → otherwise begin SERP/title/snippet review → consider content refresh only after human review.**

In [ ]:
# Import modules needed to identify the Colab environment,
# clone the repository, work with files, and load the CSV.
import os
import sys
import subprocess
import pandas as pd


# Check whether the notebook is running in Google Colab.
# This allows the setup to work differently if the notebook
# is later run outside Colab.
IN_COLAB = "google.colab" in sys.modules


# Define the public GitHub repository containing the
# validated Week-6 output.
REPO_URL = "https://github.com/innouguru/flyrank-internship-ml"
REPO_DIR = "flyrank-internship-ml"


# When running in Colab, clone the repository into the
# current runtime if it has not already been cloned.
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )

    # Change the working directory to the repository root
    # so that repository files can be accessed using
    # relative paths.
    os.chdir(REPO_DIR)


# Confirm the directory from which the notebook is working.
print("Working dir:", os.getcwd())


# Define the location of the validated Week-6 output.
# This file contains the 50 highest-ranked February 2026
# candidates selected using the validated Random Forest.
input_file = "work/outputs/top_50_recommendation.csv"


# Stop the notebook if the expected validated output
# cannot be found.
assert os.path.exists(
    input_file
), "top_50_recommendation.csv not found."


# Load the validated Week-6 ranking into a DataFrame.
top_50 = pd.read_csv(input_file)


# Confirm that the expected output was loaded successfully
# and inspect its size and available fields.
print("Validated Week-6 queue loaded.")
print("Rows:", len(top_50))
print("Columns:")
print(top_50.columns.tolist())

Working dir: /content/flyrank-internship-ml/flyrank-internship-ml
Validated Week-6 queue loaded.
Rows: 50
Columns:
['month', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_engaged_sessions', 'ctr', 'position_start', 'peer_median_ctr', 'peer_count', 'ctr_below_position_peers', 'next_month', 'next_ctr', 'next_peer_median_ctr', 'next_peer_count', 'future_ctr_improved', 'target', 'model_score']


In [ ]:
# Inspect the first few rows of the validated Week-6 queue
# to understand the information available for the action playbook.

display(top_50.head(10))

,month,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions,ctr,position_start,peer_median_ctr,peer_count,ctr_below_position_peers,next_month,next_ctr,next_peer_median_ctr,next_peer_count,future_ctr_improved,target,model_score
0,2026-02-01,client_e547b89c05043229,content_a4c97a728f264492,10813.0,78.0,4.125035,93.0,5.0,0.721354,1.0,0.750221,4149,True,2026-03-01,0.436872,0.581395,36634,False,0,0.918765
1,2026-02-01,client_e547b89c05043229,content_7d85af9ae830df6c,10192.0,73.0,4.820742,84.0,11.0,0.716248,1.0,0.750221,4149,True,2026-03-01,0.618047,0.581395,36634,True,1,0.903497
2,2026-02-01,client_e547b89c05043229,content_9d82eac28dba686f,149.0,1.0,5.107383,3.0,0.0,0.671141,1.0,0.750221,4149,True,2026-03-01,1.578947,0.342466,11467,True,1,0.900294
3,2026-02-01,client_e547b89c05043229,content_9206a47f6a5a9d1f,1786.0,13.0,8.329227,17.0,1.0,0.727884,1.0,0.750221,4149,True,2026-03-01,0.946021,0.342466,11467,True,1,0.891390
4,2026-02-01,client_e547b89c05043229,content_9489c3425cacc5d8,1796.0,13.0,6.128619,20.0,2.0,0.723831,1.0,0.750221,4149,True,2026-03-01,0.470773,0.581395,36634,False,0,0.885218
5,2026-02-01,client_e547b89c05043229,content_1d00260ae6e4f9ed,9760.0,65.0,5.555738,81.0,2.0,0.665984,1.0,0.750221,4149,True,2026-03-01,0.749918,0.581395,36634,True,1,0.884758
6,2026-02-01,client_f623b01661d4bfe4,content_9e5756c79cdca7df,89.0,9.0,42.056180,21.0,1.0,10.112360,41.0,10.611735,16,True,2026-03-01,3.661972,0.000000,3691,True,1,0.876507
7,2026-02-01,client_f623b01661d4bfe4,content_369b08251e576487,1552.0,21.0,7.446521,39.0,2.0,1.353093,1.0,33.333333,83,True,2026-03-01,1.914414,0.581395,36634,True,1,0.876041
8,2026-02-01,client_e547b89c05043229,content_96c3f5ee733e3ca2,148.0,1.0,4.810811,2.0,0.0,0.675676,1.0,0.750221,4149,True,2026-03-01,0.759494,0.581395,36634,True,1,0.876026
9,2026-02-01,client_e547b89c05043229,content_47c440ee3423478c,158.0,1.0,4.537975,3.0,0.0,0.632911,1.0,0.750221,4149,True,2026-03-01,1.298701,0.581395,36634,True,1,0.872291


In [ ]:
# Show the lowest-exposure recommendations so we can
# examine how much search evidence they have.

display(
    top_50[
        [
            "content_hash_id",
            "gsc_impressions",
            "gsc_clicks",
            "ctr",
            "gsc_avg_position",
            "model_score"
        ]
    ]
    .sort_values("gsc_impressions")
    .head(10)
)

,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,model_score
41,content_d359e26e27260345,19.0,3.0,15.789474,9.526316,0.808265
37,content_7d8c5d6c5fae7bd4,24.0,2.0,8.333333,42.208333,0.811954
23,content_410bdadcf0e9d22e,30.0,2.0,6.666667,12.933333,0.841224
35,content_d51c6315f9c34ab0,32.0,2.0,6.250000,26.812500,0.812494
29,content_a23309f704c98e59,35.0,4.0,11.428571,5.457143,0.824566
39,content_16949d8f63c6f458,78.0,2.0,2.564103,11.102564,0.811823
16,content_eb90d454afa7daaa,82.0,4.0,4.878049,13.560976,0.860678
6,content_9e5756c79cdca7df,89.0,9.0,10.112360,42.056180,0.876507
15,content_bf5762e2e666f20d,92.0,2.0,2.173913,43.173913,0.860740
43,content_ffe5168579081af3,135.0,1.0,0.740741,8.148148,0.807170


In [ ]:
# Group the recommendations by search exposure.
# The bands help us see how many pages have limited evidence
# before setting a monitoring threshold.

impression_bands = pd.cut(
    top_50["gsc_impressions"],
    bins=[0, 50, 100, 250, 500, 1000, float("inf")],
    labels=[
        "<=50",
        "51-100",
        "101-250",
        "251-500",
        "501-1000",
        ">1000"
    ]
)

print(
    impression_bands.value_counts()
    .sort_index()
)

gsc_impressions
<=50         5
51-100       4
101-250     16
251-500      2
501-1000     1
>1000       22
Name: count, dtype: int64


In [ ]:
# Calculate the proportion of the top-50 queue that falls
# below several possible exposure thresholds.

for threshold in [50, 100, 250, 500, 1000]:
    count = (top_50["gsc_impressions"] <= threshold).sum()
    percentage = count / len(top_50) * 100

    print(
        f"<= {threshold} impressions: "
        f"{count} pages ({percentage:.1f}%)"
    )

<= 50 impressions: 5 pages (10.0%)
<= 100 impressions: 9 pages (18.0%)
<= 250 impressions: 25 pages (50.0%)
<= 500 impressions: 27 pages (54.0%)
<= 1000 impressions: 28 pages (56.0%)


In [ ]:
# Create a copy of the validated Week-6 recommendations
# so that the original output is not modified.
action_queue = top_50.copy()


# Rank the recommendations from highest to lowest
# predicted probability of CTR recovery.
action_queue = action_queue.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

action_queue["rank"] = range(1, len(action_queue) + 1)


# Keep the Week-4 reason code because it explains why
# each page entered the candidate pool.
action_queue["reason_code"] = (
    "CTR_BELOW_POSITION_PEERS"
)


# Assign the initial action based on search exposure.
#
# Pages with 100 or fewer impressions have limited search
# evidence, so they are monitored rather than immediately
# sent for an intervention.
#
# Pages above 100 impressions receive an initial SERP review.
action_queue["recommended_action"] = (
    action_queue["gsc_impressions"]
    .apply(
        lambda impressions:
            "MONITOR"
            if impressions <= 100
            else "SERP_REVIEW"
    )
)


# Add a short explanation for the assigned action.
action_queue["action_reason"] = (
    action_queue["gsc_impressions"]
    .apply(
        lambda impressions:
            "Limited search exposure; gather more evidence before intervention."
            if impressions <= 100
            else "Sufficient search exposure for human review of search intent and SERP presentation."
    )
)


# Select the fields needed for the human-facing queue.
queue_columns = [
    "rank",
    "content_hash_id",
    "model_score",
    "reason_code",
    "recommended_action",
    "action_reason",
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "peer_median_ctr",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions"
]

action_queue = action_queue[queue_columns]


# Check the resulting queue.
print("Action queue rows:", len(action_queue))
print("\nAction counts:")
print(action_queue["recommended_action"].value_counts())

display(action_queue.head(10))

Action queue rows: 50

Action counts:
recommended_action
SERP_REVIEW    41
MONITOR         9
Name: count, dtype: int64


,rank,content_hash_id,model_score,reason_code,recommended_action,action_reason,gsc_impressions,gsc_clicks,ctr,peer_median_ctr,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions
0,1,content_a4c97a728f264492,0.918765,CTR_BELOW_POSITION_PEERS,SERP_REVIEW,Sufficient search exposure for human review of...,10813.0,78.0,0.721354,0.750221,4.125035,93.0,5.0
1,2,content_7d85af9ae830df6c,0.903497,CTR_BELOW_POSITION_PEERS,SERP_REVIEW,Sufficient search exposure for human review of...,10192.0,73.0,0.716248,0.750221,4.820742,84.0,11.0
2,3,content_9d82eac28dba686f,0.900294,CTR_BELOW_POSITION_PEERS,SERP_REVIEW,Sufficient search exposure for human review of...,149.0,1.0,0.671141,0.750221,5.107383,3.0,0.0
3,4,content_9206a47f6a5a9d1f,0.891390,CTR_BELOW_POSITION_PEERS,SERP_REVIEW,Sufficient search exposure for human review of...,1786.0,13.0,0.727884,0.750221,8.329227,17.0,1.0
4,5,content_9489c3425cacc5d8,0.885218,CTR_BELOW_POSITION_PEERS,SERP_REVIEW,Sufficient search exposure for human review of...,1796.0,13.0,0.723831,0.750221,6.128619,20.0,2.0
5,6,content_1d00260ae6e4f9ed,0.884758,CTR_BELOW_POSITION_PEERS,SERP_REVIEW,Sufficient search exposure for human review of...,9760.0,65.0,0.665984,0.750221,5.555738,81.0,2.0
6,7,content_9e5756c79cdca7df,0.876507,CTR_BELOW_POSITION_PEERS,MONITOR,Limited search exposure; gather more evidence ...,89.0,9.0,10.112360,10.611735,42.056180,21.0,1.0
7,8,content_369b08251e576487,0.876041,CTR_BELOW_POSITION_PEERS,SERP_REVIEW,Sufficient search exposure for human review of...,1552.0,21.0,1.353093,33.333333,7.446521,39.0,2.0
8,9,content_96c3f5ee733e3ca2,0.876026,CTR_BELOW_POSITION_PEERS,SERP_REVIEW,Sufficient search exposure for human review of...,148.0,1.0,0.675676,0.750221,4.810811,2.0,0.0
9,10,content_47c440ee3423478c,0.872291,CTR_BELOW_POSITION_PEERS,SERP_REVIEW,Sufficient search exposure for human review of...,158.0,1.0,0.632911,0.750221,4.537975,3.0,0.0


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended Use**

The playbook is intended for editors or content teams to identify the top 50 pages that may need attention. The ranking helps the team prioritize its limited review capacity by bringing potentially important CTR opportunities to the top of the queue.

**Limits**

The recommendations are **decision-support, not final verdicts**. A high model score does not mean that a page should automatically be changed. Editors must review the page and use their judgement to decide whether the page actually needs attention and what type of work is appropriate.

The playbook is limited to the signals available to the validated model and the `CTR_BELLOW_POSITION_PEER` candidate rule. It does not establish why a page has low CTR, and it does not directly evaluate the quality, relevance, accuracy, or usefulness of the content.

The recommended `SERP_REVIEW` action therefore means that the page should be investigated, starting with the title, snippet, and search-intent alignment. A deeper content refresh should only be considered after human review.

Pages with limited search exposure are assigned `MONITOR` because there may not yet be enough evidence to justify an intervention.

The playbook should not be treated as a production system or as an automated content-editing system. Its purpose is to help humans decide **which pages to review first**, while leaving the final decision and the type of intervention to the editor.

In [ ]:
# Check that the action queue contains the expected
# number of recommendations and intended action types.

print("Recommendations in queue:", len(action_queue))

print("\nRecommended actions:")
print(
    action_queue["recommended_action"]
    .value_counts()
)

# Confirm that the queue contains exactly 50 recommendations.
assert len(action_queue) == 50

# Confirm that only the intended initial actions
# are present in the queue.
allowed_actions = {
    "SERP_REVIEW",
    "MONITOR"
}

assert set(
    action_queue["recommended_action"].unique()
).issubset(allowed_actions)

print("\nIntended-use checks passed.")

Recommendations in queue: 50

Recommended actions:
recommended_action
SERP_REVIEW    41
MONITOR         9
Name: count, dtype: int64

Intended-use checks passed.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

The ranked queue is intended to support human review. Before acting on a recommendation, an editor should check the page's search intent, title, snippet, and the relationship between the search result and the content that users find after clicking.

For pages assigned `SERP_REVIEW`, the editor should first determine whether the title and snippet accurately communicate the page's value and match the likely search intent. The editor should also consider the available engagement signals before deciding whether a deeper content review is necessary.

For pages assigned `MONITOR`, the editor should avoid making an immediate change based only on the model recommendation. The page should first be allowed to accumulate more search evidence.

The playbook does not determine the exact cause of low CTR or the correct editorial intervention. The editor is responsible for deciding whether a change is justified and what type of change is appropriate.

#### No-go cases

The following decisions should not be automated from this playbook:

- Treating a high model score as proof that a page needs a content change.
- Assuming that low CTR is caused by poor content.
- Automatically deciding that a page is outdated, inaccurate, or irrelevant.
- Automatically deciding that a page should be refreshed, rewritten, or removed.
- Automatically changing the search intent of a page.
- Publishing or approving editorial changes without human review.
- Treating the recommendations as a final verdict rather than decision-support.

The model's role ends at prioritization and recommendation. Human judgement remains responsible for the final editorial decision.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The recommendations should be checked on later data to determine whether the model remains useful. Monitoring is focused on whether the ranking performance and the underlying data patterns remain reasonably consistent with the validation results.

The main performance check is **Precision@50** on a later period where the future outcome is available. A sustained decline in Precision@50 would indicate that the ranking may no longer be prioritizing recovery opportunities as effectively.

The input data should also be monitored for meaningful changes in the distributions of impressions, CTR, search position, and engagement signals. Large changes may indicate that the conditions under which the model was validated no longer represent the current data.

The relationship between the `CTR_BELOW_POSITION_PEERS` candidate rule and future CTR recovery should also be checked. If this relationship changes substantially, the current candidate definition or model may no longer be appropriate.

Retraining should be considered when later evaluation shows sustained degradation in ranking performance or when meaningful changes in the input data or recovery relationship are observed. Retraining should be followed by the same validation process rather than being triggered only by the passage of time.

These are light monitoring and retrain triggers for a research decision-support workflow, not a production monitoring system.

In [ ]:
# Calculate the current Precision@50 from the validated
# Week-6 queue so that Week 7 has a baseline for future
# monitoring.

baseline_precision_at_50 = top_50["target"].mean()

print(
    "Validated February Precision@50:",
    round(baseline_precision_at_50, 4)
)

print(
    "\nFuture monitoring should compare later-period "
    "Precision@50 against this validated baseline."
)

Validated February Precision@50: 0.8

Future monitoring should compare later-period Precision@50 against this validated baseline.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The final action queue is exported as a CSV so that the ranked recommendations can be reused in the research paper.

The export contains the model ranking, reason code, recommended starting action, and the search and engagement signals used to provide context for human review.

Future outcome fields such as `target`, `future_ctr_improved`, `next_ctr`, and other next-month measurements are excluded from the playbook export because they are evaluation information rather than information available when making a recommendation.

The exported queue contains hashed content identifiers rather than client names or private queries.

In [ ]:
# Define the output directory used by the internship repository.
output_dir = "work/outputs"

# Create the directory if it does not already exist.
os.makedirs(output_dir, exist_ok=True)


# Define the fields that should be included in the
# human-facing action queue and research-paper output.
export_columns = [
    "rank",
    "content_hash_id",
    "model_score",
    "reason_code",
    "recommended_action",
    "action_reason",
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "peer_median_ctr",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions"
]


# Create the final export without evaluation-only fields
# such as target and future-month outcomes.
paper_queue = action_queue[export_columns].copy()


# Save the ranked action queue as a CSV file.
output_file = os.path.join(
    output_dir,
    "week7_action_queue.csv"
)

paper_queue.to_csv(
    output_file,
    index=False
)


# Confirm that the export was created successfully.
print("Exported:", output_file)
print("Rows:", len(paper_queue))
print("Columns:", paper_queue.columns.tolist())

Exported: work/outputs/week7_action_queue.csv
Rows: 50
Columns: ['rank', 'content_hash_id', 'model_score', 'reason_code', 'recommended_action', 'action_reason', 'gsc_impressions', 'gsc_clicks', 'ctr', 'peer_median_ctr', 'gsc_avg_position', 'ga4_pageviews', 'ga4_engaged_sessions']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.